In [0]:
# ── CELL 1 — BRONZE LAYER ─────────────────────────────────────────
# Read raw data as-is — no transformations
# This is your source of truth

df_bronze_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/basic/default/test/orders.csv")

df_bronze_customers = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/basic/default/test/customers.csv")

df_bronze_products = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/basic/default/test/products.csv")

# Save Bronze tables as-is
df_bronze_orders.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.bronze_orders")

df_bronze_customers.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.bronze_customers")

df_bronze_products.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.bronze_products")

print(f"✅ Bronze Orders    : {df_bronze_orders.count():,} rows")
print(f"✅ Bronze Customers : {df_bronze_customers.count():,} rows")
print(f"✅ Bronze Products  : {df_bronze_products.count():,} rows")
print("Bronze layer saved successfully!")

✅ Bronze Orders    : 300,000 rows
✅ Bronze Customers : 50,000 rows
✅ Bronze Products  : 352 rows
Bronze layer saved successfully!


In [0]:
from pyspark.sql.functions import *

Orders_cleaned = df_bronze_orders \
    .dropDuplicates(["OrderID"]) \
    .dropna(subset=["OrderID", "CustomerID", "ProductID","NetAmount","OrderDate"]) \
    .withColumn("Status", upper(col("Status"))) \
    .filter(col("Status") != "CANCELLED")\
    .withColumn("Orders_ProcessedAt", current_timestamp())

print(f"Orders_cleaned: {Orders_cleaned.count():,} rows")
print(f"Raw: {df_bronze_orders.count():,} rows")


Orders_cleaned: 262,568 rows
Raw: 300,000 rows


In [0]:
Products_cleaned = df_bronze_products \
    .dropDuplicates(["ProductID"]) \
    .dropna(subset=["ProductID", "ProductName", "Category"]) \
    .withColumn("Category",      trim(initcap(col("Category")))) \
    .withColumn("Brand",         trim(initcap(col("Brand")))) \
    .withColumnRenamed("Status", "ProductStatus") \
    .withColumn("ProductStatus", upper(trim(col("ProductStatus")))) \
    .filter(col("ProductStatus") != "INACTIVE") \
    .withColumn("Products_ProcessedAt", current_timestamp())

print(f"Raw products    : {df_bronze_products.count():,}")
print(f"Cleaned products: {Products_cleaned.count():,}")


Raw products    : 352
Cleaned products: 263


In [0]:
from pyspark.sql.functions import *

Customers_cleaned = df_bronze_customers \
    .dropDuplicates(["CustomerID"]) \
    .dropna(subset=["CustomerID", "FirstName", "LastName"]) \
    .fillna({"City": "Unknown", "CustomerSegment": "Unknown"}) \
    .withColumn("FirstName",trim(initcap(col("FirstName")))) \
    .withColumn("LastName",trim(initcap(col("LastName")))) \
    .withColumn("City",trim(initcap(col("City")))) \
    .withColumn("State",trim(initcap(col("State")))) \
    .withColumn("CustomerSegment",trim(initcap(col("CustomerSegment")))) \
    .withColumn("Gender",trim(initcap(col("Gender")))) \
    .withColumn("Email",trim(col("Email"))) \
    .withColumn("RegistrationDate",to_date(col("RegistrationDate"), "yyyy-MM-dd")) \
    .withColumn("IsActive",upper(trim(col("IsActive")))) \
    .filter(col("IsActive") != "NO") \
    .filter((col("Age") >= 18) & (col("Age") <= 80)) \
    .withColumn("Customers_ProcessedAt",current_timestamp())

print(f"Raw customers    : {df_bronze_customers.count():,}")
print(f"Cleaned customers: {Customers_cleaned.count():,}")

Raw customers    : 50,000
Cleaned customers: 37,515


In [0]:
from pyspark.sql.functions import *

df_orders_customers = Orders_cleaned.join(
    Customers_cleaned, ["CustomerID"], "left"
)
Salesdata = df_orders_customers.join(
    Products_cleaned, ["ProductID"], "left"
)

print(f"Silver master rows: {Salesdata.count():,}")
Salesdata.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.Salesdata")



Silver master rows: 262,568


In [0]:
Salesdata.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.Salesdata")

In [0]:
data = spark.read.table("basic.default.Salesdata")
display(data.limit(10))

ProductID,CustomerID,OrderID,OrderDate,Quantity,UnitPrice,DiscountPct,DiscountAmount,GrossAmount,NetAmount,PaymentMode,Channel,Status,ShippingCity,ShippingState,DeliveryDate,DeliveryDays,ReturnFlag,UpdatedAt,Orders_ProcessedAt,FirstName,LastName,Email,Gender,Age,City,State,CustomerSegment,RegistrationDate,IsActive,Customers_ProcessedAt,ProductName,Category,Brand,Price,CostPrice,Rating,ReviewCount,ProductStatus,Products_ProcessedAt
385,39547,161,2021-01-26,2,46132.43,10,9226.49,92264.86,83038.37,Cash on Delivery,Website,PROCESSING,Nizamabad,Telangana,2021-02-01,6,0,2021-01-26T00:00:00.000Z,2026-03-21T22:35:46.662Z,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
314,12661,295,2021-12-24,3,34702.4,0,0.0,104107.2,104107.2,UPI,Tablet,DELIVERED,Kota,Rajasthan,2021-12-31,7,0,2021-12-24T00:00:00.000Z,2026-03-21T22:35:46.662Z,Suresh,Agarwal,suresh.agarwal12661@email.com,Female,36,Mumbai,Maharashtra,Vip,2022-12-04,YES,2026-03-21T22:35:46.662Z,Adidas Almonds,Grocery,Adidas,1373.51,643.33,4.0,4514,ACTIVE,2026-03-21T22:35:46.662Z
141,29298,340,2021-03-02,1,1697.58,0,0.0,1697.58,1697.58,Credit Card,Mobile App,RETURNED,Vadodara,Gujarat,2021-03-06,4,1,2021-03-02T00:00:00.000Z,2026-03-21T22:35:46.662Z,null,null,null,null,null,null,null,null,null,null,null,Nike Vacuum Cleaner,Home & Kitchen,Nike,6019.19,2883.07,4.5,4225,ACTIVE,2026-03-21T22:35:46.662Z
377,104,536,2023-01-23,5,40341.9,10,20170.95,201709.5,181538.55,Wallet,Mobile App,DELIVERED,Chennai,Tamil Nadu,2023-01-29,6,0,2023-01-23T00:00:00.000Z,2026-03-21T22:35:46.662Z,Sneha,Patel,sneha.patel104@email.com,Male,37,Jodhpur,Rajasthan,Regular,2020-01-19,YES,2026-03-21T22:35:46.662Z,null,null,null,null,null,null,null,null,null
225,13368,546,2024-09-19,1,1223.15,0,0.0,1223.15,1223.15,UPI,Website,SHIPPED,Rajkot,Gujarat,2024-09-28,9,0,2024-09-19T00:00:00.000Z,2026-03-21T22:35:46.662Z,Lakshmi,Nair,lakshmi.nair13368@email.com,Male,63,Asansol,West Bengal,Premium,2023-01-10,YES,2026-03-21T22:35:46.662Z,OnePlus Badminton Racket,Sports,Oneplus,1590.78,757.95,3.9,3290,ACTIVE,2026-03-21T22:35:46.662Z
464,42579,612,2022-10-16,2,35304.73,0,0.0,70609.46,70609.46,Debit Card,Tablet,SHIPPED,Coimbatore,Tamil Nadu,2022-10-26,10,0,2022-10-16T00:00:00.000Z,2026-03-21T22:35:46.662Z,Manoj,Joshi,manoj.joshi42579@email.com,Male,61,Jodhpur,Rajasthan,Regular,2021-03-02,YES,2026-03-21T22:35:46.662Z,null,null,null,null,null,null,null,null,null
276,8225,648,2022-07-13,5,981.41,5,245.35,4907.05,4661.7,Debit Card,Website,PROCESSING,Dwarka,Delhi,2022-07-20,7,0,2022-07-13T00:00:00.000Z,2026-03-21T22:35:46.662Z,Rajesh,Menon,rajesh.menon8225@email.com,Female,23,Howrah,West Bengal,Regular,2021-07-13,YES,2026-03-21T22:35:46.662Z,Butterfly Eye Liner,Beauty,Butterfly,240.31,133.64,3.9,1488,ACTIVE,2026-03-21T22:35:46.662Z
193,40891,668,2021-09-12,1,1698.46,25,424.62,1698.46,1273.84,Credit Card,Mobile App,DELIVERED,Howrah,West Bengal,2021-09-13,1,0,2021-09-12T00:00:00.000Z,2026-03-21T22:35:46.662Z,Ganesh,Gupta,ganesh.gupta40891@email.com,Male,42,Mangalore,Karnataka,Premium,2019-06-10,YES,2026-03-21T22:35:46.662Z,Sony Comic Book,Books,Sony,297.32,135.71,3.0,1959,ACTIVE,2026-03-21T22:35:46.662Z
479,5320,777,2024-06-26,3,40040.92,10,12012.28,120122.76,108110.48,Cash on Delivery,Website,DELIVERED,Nagpur,Maharashtra,2024-07-04,8,0,2024-06-26T00:00:00.000Z,2026-03-21T22:35:46.662Z,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
88,39645,828,2023-10-04,3,829.09,0,0.0,2487.27,2487.27,Wallet,Mobile App,DELIVERED,Aurangabad,Maharashtra,2023-10-11,7,0,2023-10-04T00:00:00.000Z,2026-03-21T22:35:46.662Z,Lakshmi,Venkatesh,lakshmi.venkatesh39645@email.com,Male,22,Rajkot,Gujarat,Regular,2020-11-16,YES,2026-03-21T22:35:46.662Z,Levi's Sandals,Clothing,Levi's,623.53,410.42,3.8,4794,ACTIVE,2026-03-21T22:35:46.662Z


In [0]:
MonthlyRevenue_SalesData = Salesdata \
    .groupBy(
        year("OrderDate").alias("Year"),
        month("OrderDate").alias("Month")
    ) \
    .agg(
        round(sum("NetAmount"), 2).alias("TotalRevenue"),
        count("OrderID").alias("TotalOrders"),
        round(avg("NetAmount"), 2).alias("AvgOrderValue")
    ) \
    .orderBy("Year", "Month")

MonthlyRevenue_SalesData.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.MonthlyRevenue_SalesData")



In [0]:
from pyspark.sql.window import Window

customersegmentation_salesdata = Salesdata \
    .groupBy("CustomerSegment") \
    .agg(
        countDistinct("CustomerID").alias("TotalCustomers"),
        count("OrderID").alias("TotalOrders"),
        round(sum("NetAmount"), 2).alias("TotalRevenue"),
        round(avg("NetAmount"), 2).alias("AvgOrderValue")
    ) \
    .orderBy("TotalRevenue", ascending=False)\
    .filter(col("CustomerSegment").isNotNull()) \
    .filter(col("CustomerSegment") != "Unknown") \
    .withColumn("RevenueRank", dense_rank().over(Window.orderBy(col("TotalRevenue").desc())))
    



customersegmentation_salesdata.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.customersegmentation_salesdata")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(customersegmentation_salesdata.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerSegment,TotalCustomers,TotalOrders,TotalRevenue,AvgOrderValue,RevenueRank
Regular,15872,83939,3.21184194819E9,38264.0,1
Premium,10502,55202,2.13294108007E9,38638.84,2
Vip,5188,27514,1.05792047131E9,38450.26,3
New,5200,27323,1.04824947646E9,38365.09,4


In [0]:
from pyspark.sql.window import Window
Product_performance_Saledata = Salesdata\
    .groupBy("ProductName","Category")\
    .agg(
        countDistinct("CustomerID").alias("TotalCustomers"),
        count("OrderID").alias("TotalOrders"),
        round(sum("NetAmount"), 2).alias("TotalRevenue"),
        round(avg("NetAmount"), 2).alias("AvgOrderValue")
    )\
    .filter(col("ProductName").isNotNull())\
    .filter(col("ProductName") != "Unknown")\
    .withColumn("CategoryRevenueRank", dense_rank().over(Window.partitionBy(col("Category")).orderBy(col("TotalRevenue").desc())))\
    .orderBy("TotalRevenue", ascending=False)\
    
Product_performance_Saledata.write.format("delta").mode("overwrite") \
    .saveAsTable("basic.default.Product_performance_Salesdata")

display(Product_performance_Saledata.limit(100))



ProductName,Category,TotalCustomers,TotalOrders,TotalRevenue,AvgOrderValue,CategoryRevenueRank
Patanjali Laptop,Electronics,540,542,1.4577612193E8,268959.63,1
Himalaya Tablet,Electronics,524,527,1.4263856559E8,270661.41,2
Adidas Tablet,Electronics,539,540,1.0692344574E8,198006.38,3
Pigeon Tablet,Electronics,531,536,1.0038751053E8,187290.13,4
Puma Smartwatch,Electronics,523,528,9.347847681E7,177042.57,5
Usha Tablet,Electronics,529,531,8.805202628E7,165823.03,6
Adidas Almonds,Grocery,1055,1071,7.528859143E7,70297.47,1
Fastrack Doll Set,Toys,1087,1096,7.389586169E7,67423.23,1
Kent Dry Fruits Mix,Grocery,1031,1047,7.005848812E7,66913.55,2
Eureka Forbes Hard Disk,Electronics,539,543,5.168748169E7,95188.73,7


In [0]:
from pyspark.sql.window import Window
region_sales_salesdata = Salesdata\
    .filter(col("ShippingState").isNotNull())\
    .filter(col("ShippingState") != "Unknown")\
    .groupBy("ShippingState", year("OrderDate").alias("Year"),month("OrderDate").alias("Month"))\
    .agg(countDistinct("CustomerID").alias("TotalCustomers"),count("OrderID").alias("TotalOrders"),\
        round(sum("NetAmount"), 2).alias("TotalRevenue"),
        round(avg("NetAmount"), 2).alias("AvgOrderValue")
    )\
    .withColumn("revenuebyState",round(sum("TotalRevenue").over(Window.partitionBy(col("ShippingState")).orderBy(col("Year"),col("Month"))),2))\
    .orderBy(asc("Year"))

display(region_sales_salesdata.limit(100))
    
region_sales_salesdata.write.format("delta").mode("overwrite").partitionBy("ShippingState") \
    .saveAsTable("basic.default.region_sales_salesdata")

ShippingState,Year,Month,TotalCustomers,TotalOrders,TotalRevenue,AvgOrderValue,revenuebyState
Tamil Nadu,2021,11,508,512,2.001284804E7,39087.59,2.3615056822E8
Telangana,2021,11,542,544,2.196508596E7,40377.0,2.3398784876E8
Rajasthan,2021,5,534,535,2.291696867E7,42835.46,1.0990773555E8
Tamil Nadu,2021,10,524,527,2.367943236E7,44932.51,2.1613772018E8
Uttar Pradesh,2021,3,560,564,1.908210444E7,33833.52,5.980242627E7
Telangana,2021,9,495,499,1.850346489E7,37081.09,1.9155934416E8
Maharashtra,2021,12,504,506,2.091137323E7,41326.82,2.4873825262E8
Rajasthan,2021,4,575,579,2.238680097E7,38664.6,8.699076688E7
Rajasthan,2021,11,530,531,2.067445829E7,38934.95,2.3207577413E8
Tamil Nadu,2021,9,567,568,2.077000349E7,36566.91,1.9245828782E8
